# 84일차 · 실험 비교와 배포 전 회귀 검사

재정렬을 켰더니 검색 점수는 올랐지만 근거가 없는 문의에도 답하기 시작했습니다. 같은 평가 질문에서 새로 생긴 실패와 비용을 함께 비교합니다.

같은 데이터 버전에서 한 요소만 바꿔 품질·지연·토큰·추정 비용을 비교한다. GitHub Actions에 도구 규칙 검사를 연결하고, 개선된 사례와 악화된 사례를 근거로 배포 또는 보류를 결정한다.

## 평균 뒤에 가려진 질문 찾기

In [ ]:
before = {"Q1": True, "Q2": False, "Q3": True}
after = {"Q1": True, "Q2": True, "Q3": False}
for q in before:
    if before[q] != after[q]:
        print(q, before[q], "→", after[q])

Q2 False → True
Q3 True → False


두 설정 모두 세 문제 중 두 문제에 정답을 냅니다. Q2는 고쳤지만 Q3는 새로 틀렸습니다. Q3가 승인 전 확정처럼 반드시 막아야 할 사례라면 평균만 보고 배포할 수 없습니다.

### 직접 해 보기

Q3를 필수 회귀 사례로 정하세요. 후보 실험의 세 요청 지연과 토큰에서 평균 지연, 총 토큰, 예시 단가 기준 추정 비용을 계산하고 배포 여부를 판정하세요.

In [ ]:
latency_ms = [240, 420, 300]
input_tokens, output_tokens = [100, 200, 300], [10, 20, 30]
# 실습용 가정 단가: 입력 100만 토큰당 1달러, 출력 100만 토큰당 4달러

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre style="white-space:pre-wrap;overflow:auto"><code>latency_ms = [240, 420, 300]
input_tokens, output_tokens = [100, 200, 300], [10, 20, 30]
baseline_ms = [200, 300, 250]
baseline_cost = (500 + 4 * 50) / 1_000_000
print(&quot;기준선&quot;, sum(baseline_ms) / len(baseline_ms), &quot;ms&quot;, 550, &quot;tokens&quot;, f&quot;${baseline_cost:.6f}&quot;)
cost = (sum(input_tokens) + 4 * sum(output_tokens)) / 1_000_000
print(sum(latency_ms) / len(latency_ms), &quot;ms&quot;, sum(input_tokens + output_tokens), &quot;tokens&quot;)
print(f&quot;${cost:.6f}&quot;, &quot;배포&quot; if after[&quot;Q3&quot;] else &quot;보류&quot;)</code></pre><pre style="white-space:pre-wrap;overflow:auto"><code>기준선 250.0 ms 550 tokens $0.000700
320.0 ms 660 tokens
$0.000840 보류
</code></pre></details>

## 수업 서비스에 적용하기

LangSmith에서 동일 데이터셋 버전의 두 실험을 열고 한 요소만 바뀌었는지 확인하세요. 각 질문의 점수와 지연, 모델 사용량을 대조하고 실제 모델 단가로 비용을 다시 계산합니다. 76일차 GitHub Actions 검사에 83일차 승인·중복 저장 규칙을 단위 테스트로 추가하세요. 승인 순서를 깨뜨린 변경이 CI에서 실패하고 복구한 커밋이 통과하는지 두 실행 링크로 확인합니다.

### 제출 전 확인

같은 데이터 버전에서 한 요소만 바꿔 품질·지연·토큰·추정 비용을 비교한다. GitHub Actions에 도구 규칙 검사를 연결하고, 개선된 사례와 악화된 사례를 근거로 배포 또는 보류를 결정한다.

[공식 문서](https://docs.langchain.com/langsmith/compare-experiment-results)